In [ ]:
import NCA
using KadanoffBaym

include("../src/wigner.jl")
include("../src/model.jl")

using SpecialFunctions: erf, erfc
using LinearAlgebra: diag, norm
using Parameters: @unpack
using ElasticArrays
using Test

using Plots, LaTeXStrings
default(fmt = :png)

In [ ]:
"""
Solve the problem in equilibrium
"""

# Single impurity Anderson model
siam = NCA.SIAM(
  V₀=sqrt(0.15),
  ε₀f=-0.4,
  tilde=true
  )

# Equilibrium NCA solver
nca = NCA.FreqNCA(
  model=siam,
  β=5e2,
  mesh=NCA.LinearMesh(-4.0,4.0,2^14-1)
    )

# Calculate Green functions
GᶜA, GᶠA, GᵇA = begin
  
  imΣ_t = NCA.solve!(nca; project=false)
  
  Ginv = NCA.slave_particle_inv_gf(nca, imΣ_t)
  Gᵇ = 1.0 ./ Ginv[1]
  Gᶠ = 1.0 ./ Ginv[2]
  
  # Trim just the imaginary part
  imGᵇ = view(reinterpret(Float64, Gᵇ), 2:2:length(Gᵇ))
  imGᵇ[imGᵇ .< nca.eps] .= 0.0
  imGᶠ = view(reinterpret(Float64, Gᶠ), 2:2:length(Gᶠ))
  imGᶠ[imGᶠ .< nca.eps] .= 0.0
  
  Gᶜ = NCA.kramerskronig(nca.mesh, π * nca.data.Nc, :real) + 1im * π * nca.data.Nc;
  
  Gᶜ, Gᶠ, Gᵇ
end;

# plot(nca.mesh.mesh, GᵇA |> imag, label=L"$A_b$", title="Spectral functions", xlab=L"$\omega$")
# plot!(nca.mesh.mesh, GᶠA |> imag, label=L"$A_f$")
# plot!(nca.mesh.mesh, GᶜA |> imag, label=L"$A_c$")
# xlims!(-0.1,0.1)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
model = PhotonAssitedModel(
  #= SIAM parameters =#
  V₀=siam.V₀,
  ε₀f=siam.ε₀f,
  λ₀=siam.λ₀,
  σ=1.0,
  
  #= Light-field parameters =#
  Γ₀=0.05,  # light-matter coupling
  ω₀=1e-1, # photon-ground state energy
  δ₀=1e-1, # photon artificial width
  
  #= Photon-pulse parameters =#
  tc=40,   # peak arrival time
  Ω=1e-1,  # frequency bandwitdh of the pulse
  
  #= SLH photon formulation =#
  Γ=5.     # pump and loss strength
)

# Pulse and SLH shape
model.ξ = t -> (model.Ω^2 / (2π))^(1/4)* exp(-(t - model.tc)^2 * (model.Ω^2 / 4))
model.γ = t -> model.Γ * model.ξ(t)
model.λ = t -> model.Γ * model.ξ(t) / sqrt(0.5 * erfc((t - model.tc) * model.Ω / sqrt(2)))
model.hᵧ= t -> (model.ω₀ - 0.5im * (model.λ(t) + model.γ(t)))

# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
data = let n₀ = 0.0 # initial photon-field occupation
  
  # Create a new frequency grid that supports for complex convolutions
  freqs = NCA.LinearMesh(nca.mesh.xmin, nca.mesh.xmax, nca.mesh.L, true);
  
  bose = NCA.nb.(freqs, nca.β)
  fermi = NCA.nf.(freqs, nca.β);
  boltz = NCA.bf.(freqs, nca.β) .|> Float64 ; boltz[boltz .== Inf] .= 0.0;
  
  # Integral over frequencies
  ∫dω(f) = trapz(freqs, f) / (2π)
  
  G₀ᶜL = ∫dω(fermi .* (GᶜA - conj(GᶜA)))
  G₀ᶜL = GreenFunction(G₀ᶜL * ones(1,1), Lesser)
  
  G₀ᶜG = -(1.0im - ∫dω(fermi .* (GᶜA - conj(GᶜA))))
  G₀ᶜG = GreenFunction(G₀ᶜG * ones(1,1), Greater)
  
  G₀ᶠL = ∫dω(fermi .* (GᶠA - conj(GᶠA)))
  G₀ᶠL = GreenFunction(G₀ᶠL * ones(1,1), Lesser)

  G₀ᶠG = -(1.0im - ∫dω(fermi .* (GᶠA - conj(GᶠA))))
  G₀ᶠG = GreenFunction(G₀ᶠG * ones(1,1), Greater)

  G₀ᵇL = -∫dω(bose .* (GᵇA - conj(GᵇA)))
  G₀ᵇL = GreenFunction(G₀ᵇL * ones(1,1), Lesser)
  
  G₀ᵇG = -(1.0im + ∫dω(bose .* (GᵇA - conj(GᵇA))))
  G₀ᵇG = GreenFunction(G₀ᵇG * ones(1,1), Greater)
  
  G₀ᵞL = -1.0im * n₀
  G₀ᵞL = GreenFunction(G₀ᵞL * ones(1,1), Lesser)
  
  G₀ᵞG = -1.0im * (1 + n₀)
  G₀ᵞG = GreenFunction(G₀ᵞG * ones(1,1), Greater)
  
  Gᶜ˥m = GreenFunction(ElasticArray(hcat(GᶜA)), MixedLesser)
  Gᶜ˥p = GreenFunction(ElasticArray(hcat(conj(GᶜA))), MixedLesser)
  
  Gᶠ˥m = GreenFunction(ElasticArray(hcat(GᶠA)), MixedLesser)
  Gᶠ˥p = GreenFunction(ElasticArray(hcat(conj(GᶠA))), MixedLesser)
  
  Gᵇ˥m = GreenFunction(ElasticArray(hcat(GᵇA)), MixedLesser)
  Gᵇ˥p = GreenFunction(ElasticArray(hcat(conj(GᵇA))), MixedLesser)
  
  # Spectral report
  println("Equilibrium spectral function normalizations:")
  println(" ∫dω Aᶜ(ω) = ", ∫dω(GᶜA - conj(GᶜA)) |> imag, ", nᶜ = ", G₀ᶜL[1,1] |> imag)
  println(" ∫dω Aᶠ(ω) = ", ∫dω(GᶠA - conj(GᶠA)) |> imag, ", nᶠ = ", G₀ᶠL[1,1] |> imag)
  println(" ∫dω Aᵇ(ω) = ", ∫dω(GᵇA - conj(GᵇA)) |> imag, ", nᵇ = ", -G₀ᵇL[1,1] |> imag)
  
  ProblemData(
    G₀ᶜL,G₀ᶜG,G₀ᶠL,G₀ᶠG,G₀ᵇL,G₀ᵇG,G₀ᵞL,G₀ᵞG,
    Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,
    freqs,fermi,bose,boltz);
  end;

In [ ]:
"""
Step until the thermal contribution is small and then discard it
"""

tspan = (0.0, 75.0)

# Criteria for evaluating the thermal collision integrals
thermal_criteria(norms) = max(norms...) > 1e-6

# Initial condition
u0 = tuple(
  [data.GᶜL,data.GᶜG,data.GᶠL,data.GᶠG,data.GᵇL,data.GᵇG,data.GᵞL,data.GᵞG],
  [data.Gᶜ˥m,data.Gᶜ˥p,data.Gᶠ˥m,data.Gᶠ˥p,data.Gᵇ˥m,data.Gᵇ˥p]
)

# Vertical step
function f_vert(u, times, t1, t2)
    return f_vert_time(model, data, times, t1, t2)
end
function f_vert_thermal(u, v, times, t1, t2)
    return f_vert(u, times, t1, t2) .+ f_vert_freq(model, data, times, t1, t2)
end

# Diagonal step
function f_diag(u, times, t1)
  println("t: $(times[t1])")
  return f_diag_time(model, data, times, t1)
end
function f_diag_thermal(u, v, times, t1)
  # TODO: Accumulate norms over horizontal step (don't take into account just the diagonal)
  t_ = f_diag_freq(model, data, times, t1)
  global thermal_norms; thermal_norms = norm.(t_)
  return f_diag(u, times, t1) .+ t_
end

# Vertical step for mixed functions
function f_line_thermal(u, v, times, t1)
  return f_line_freq(model, data, times, t1)
end

# The solver will stop when tspan[2] is reached or `callback` evaluates to `false`
state, caches = kbsolve(f_vert_thermal, f_diag_thermal, u0, tspan[1], tspan[2]; f_line=f_line_thermal,
    dt=1e-4, max_dt=0.05, atol=1e-9, rtol=1e-7, callback=()->thermal_criteria(thermal_norms))

@testset "KMS boundary conditions" begin
  ≅(a,b) = isapprox(a,b;rtol=1e-2)
  
  @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,fermi,bose,freqs = data
  ∫dω(f) = trapz(freqs, f) / (2π)
  
  # G^R(t,t) = -i
#   @test all(diag(GᶜG.data) - diag(GᶜL.data) .≅ -1.0im)
#   @test all(diag(GᶠG.data) - diag(GᶠL.data) .≅ -1.0im)
#   @test all(diag(GᵇG.data) - diag(GᵇL.data) .≅ -1.0im)
#   @test all(diag(GᵞG.data) - diag(GᵞL.data) .≅ -1.0im)
  
  # C^<(t,0) = C˥(t,0+)
  @test all(GᶜL[t,1] ≅ ∫dω(fermi .* (Gᶜ˥m[t] - Gᶜ˥p[t])) for t in 1:size(Gᶜ˥m,2))
  @test all(GᶠL[t,1] ≅ ∫dω(fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in 1:size(Gᶠ˥m,2))
  @test all(GᵇL[t,1] ≅ ∫dω(-bose .* (Gᵇ˥m[t] - Gᵇ˥p[t])) for t in 1:size(Gᵇ˥m,2))
end

# If we don't need to evaluate the thermal collision integrals, we need to manually tune some parameters
if !thermal_criteria(thermal_norms)
  
  # The mixed functions are 1 time-step longer than they should be since a new timestep has been proposed
  resize!.(state.u[2], length(state.t) - 1)
  
  # We need a new integrator-state without the mixed functions (a new timestep has already been proposed)
  state = KadanoffBaym.VCABMState((state.u[1],), state.t[1:end-1], state.t[end] - state.t[end-1])
  
  state, _ = KadanoffBaym.kbsolve_(state, caches, f_vert, f_diag, nothing, tspan[2], true,
 0.2, 1e-9, 1e-7, 12, 5, 1//5, 9//10, ()->true)
end

# state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2]; max_dt=0.05, atol=1e-9, rtol=1e-7)

# Extract info
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,freqs = data;
times = state.t;

In [ ]:
# Pulse
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, model.ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
# Mixed spectral functions
plot(nca.mesh.mesh, Gᶜ˥m.data[:,1] - Gᶜ˥p.data[:,1] |> imag)
plot!(nca.mesh.mesh, Gᶜ˥m.data[:,end] - Gᶜ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᵇ˥m.data[:,1] - Gᵇ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᵇ˥m.data[:,end] - Gᵇ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᶠ˥m.data[:,1] - Gᶠ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᶠ˥m.data[:,end] - Gᶠ˥p.data[:,end] |> imag .|> -)

In [ ]:
# Slave-particle occupation
plot(times,-imag(diag(GᵇL.data)))
plot!(twinx(),times,imag(diag(GᶠL.data)), color=:orange)

In [ ]:
# Time-dependent spectral functions
ρ = wigner_transform(GᵞG.data - GᵞL.data)
wigner_plot!(real(ρ), times)

# ρ = wigner_transform(GᵇG.data - GᵇL.data)#; Nf=4096*2);
# wigner_plot!(real(ρ), times)#; ylims=(-10,10))

# ρ = wigner_transform(GᶠG.data - GᶠL.data)#; Nf=4096*2);
# wigner_plot!(real(ρ), times)#; ylims=(-10,10))